[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Dataclasses &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

Run the cell below first.


In [1]:
from dataclasses import dataclass, field, asdict, replace, FrozenInstanceError
import json

print("ready")


ready


**1.** A book as a dataclass.


In [2]:
@dataclass
class Book:
    title: str
    author: str
    pages: int


dune = Book("Dune", "Herbert", 412)

print(dune)
print(Book("Dune", "Herbert", 412) == dune)


Book(title='Dune', author='Herbert', pages=412)
True


Three lines of fields replace the `__init__`, `__repr__` and `__eq__` that the **Dunder Methods**
exercises wrote out by hand.


**2.** A list field that each book gets its own copy of.


In [3]:
@dataclass
class Book:
    title: str
    author: str
    pages: int
    tags: list[str] = field(default_factory=list)


dune = Book("Dune", "Herbert", 412)
emma = Book("Emma", "Austen", 474)
dune.tags.append("science fiction")

print(dune.tags, emma.tags)


['science fiction'] []


`tags: list[str] = []` would have been refused when the class was defined. `default_factory=list`
calls `list()` once per book, so the two books hold different lists.


**3.** A check in `__post_init__`.


In [4]:
@dataclass
class Book:
    title: str
    author: str
    pages: int

    def __post_init__(self):
        if self.pages < 1:
            raise ValueError(f"a book needs at least one page, not {self.pages}")


try:
    Book("Untitled", "Nobody", 0)
except ValueError as error:
    print("refused:", error)


refused: a book needs at least one page, not 0


The generated `__init__` stored all three fields and then called `__post_init__`, which raised before
the object was handed back. Nothing ever received a book with no pages.


**4.** Equality on title and author only.


In [5]:
@dataclass
class Book:
    title: str
    author: str
    pages: int = field(compare=False)


first_edition = Book("Dune", "Herbert", 412)
reprint = Book("Dune", "Herbert", 528)

print(first_edition == reprint)
print(first_edition, reprint)


True
Book(title='Dune', author='Herbert', pages=412) Book(title='Dune', author='Herbert', pages=528)


The two editions are equal because `pages` is left out of `__eq__`, and both printouts still show
their own page counts, because `repr` was not turned off.


**5.** A frozen book.


In [6]:
@dataclass(frozen=True)
class Book:
    title: str
    author: str
    pages: int = field(compare=False)


dune = Book("Dune", "Herbert", 412)

try:
    dune.pages = 500
except FrozenInstanceError as error:
    print("refused:", error)

print("in a set:", {dune, Book("Dune", "Herbert", 528)})
print("replace: ", replace(dune, pages=528))


refused: cannot assign to field 'pages'
in a set: {Book(title='Dune', author='Herbert', pages=412)}
replace:  Book(title='Dune', author='Herbert', pages=528)


The set holds one book, because the hash, like `==`, uses only the fields that are compared, and the
two books agree on those. `replace` built a new book and left `dune` unchanged, which is the only way
to get a different page count from a frozen object.


**6.** JSON and back.


In [7]:
@dataclass
class Book:
    title: str
    author: str
    pages: int


dune = Book("Dune", "Herbert", 412)

text = json.dumps(asdict(dune))
copy = Book(**json.loads(text))

print(text)
print(copy)
print(copy == dune)


{"title": "Dune", "author": "Herbert", "pages": 412}
Book(title='Dune', author='Herbert', pages=412)
True


`asdict` gave a dictionary with the field names as keys, and `**` spread those keys back out as keyword
arguments, so the copy was built exactly as `Book(title="Dune", author="Herbert", pages=412)` would
build it.


---

&#8592; **Back to:** [Dataclasses](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/11-dataclasses.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
